# State-Conditioned Rebound/Risk Router v1

Direction: `reward` / `selection` diagnostic.  
Role: `diagnose`, not a production promotion.  
Changed layer: fixed-path counterfactual that approximates a state-conditioned reward or auxiliary critic before any PPO retrain.

## Baselines

- Canonical control family: `balanced_rule_residual` descendants.
- Direct safety base: `risk_budget_residual_drawdown_guard_v1`, because it already clears the strict OOS target (`relative_return >= 30pp`, `max_dd < 15%`) and has saved walk-forward daily paths.
- Upside candidates: existing recovery/re-entry execution notebooks with saved strict and walk-forward daily paths.

## Historical reflection

- Closest success: residual drawdown guard and predictive drawdown gate show strict OOS can clear `+30pp` / `<15% MaxDD` when exposure is low and drawdown-aware.
- Closest failure: scalar reward proxy and rebound-only/global-cap screens improve one window while failing walk-forward drawdown or rebound capture.
- Why this is not a duplicate: this notebook tests a dual-state router, not another global scalar weight or cap. It uses state-specific behavior: downtrend/broken-relative-risk caps and high-volatility positive-momentum rebound lifts.

## External basis

- Qlib keeps the research loop inside a reproducible AI quant workflow and supports RL for continuous investment decisions: https://github.com/microsoft/qlib
- DeepTrader motivates market-condition embedding and negative maximum drawdown for risk-return balancing: https://ojs.aaai.org/index.php/AAAI/article/view/16144
- Expected maximum drawdown RRL motivates path-dependent downside-risk reward objectives: https://www.sciencedirect.com/science/article/pii/S0957417417304402
- FinRL motivates separating market environment, agent, and financial application/risk controls: https://github.com/AI4Finance-Foundation/FinRL

## Hypothesis

A state-conditioned reward should not penalize drawdown or reward rebound globally. It should:

1. cap exposure in broken-anchor / weak-momentum / SOXL-vs-SOXX stress states;
2. lift exposure only in high-volatility positive-momentum rebound states;
3. be selected by strict OOS plus walk-forward guards, not strict return alone.

Acceptance gate for full pass:

- strict OOS `relative_return >= 30pp` and `max_dd <= 15%`;
- each saved walk-forward window `max_dd <= 15%`;
- walk-forward relative-return floors: `wf_2020 >= -10pp`, `wf_2022 >= 30pp`, `wf_2023_2024 >= 30pp`.


In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "runtime").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Could not locate repository root containing AGENTS.md and runtime/")
CARD = ROOT / "runtime" / "reward" / "card" / "state_conditioned_rebound_risk_router_v1"
CARD.mkdir(parents=True, exist_ok=True)

STARTING_NAV = 1_000_000.0
FULL_PASS_REL_FLOORS = {
    "wf_2020": -10.0,
    "wf_2022": 30.0,
    "wf_2023_2024": 30.0,
}
WINDOWS = ["strict_oos", "wf_2020", "wf_2022", "wf_2023_2024"]

BASE_PATHS = {
    "strict_oos": ROOT / "runtime/execution/card/risk_budget_residual_drawdown_guard_v1/risk_budget_residual_drawdown_guard_v1_selected_soxl_daily.csv",
    "wf_2020": ROOT / "runtime/execution/card/risk_budget_residual_drawdown_guard_walkforward_v1/walkforward/risk_budget_residual_drawdown_guard_walkforward_v1_wf_2020_residual_guard_soxl_daily.csv",
    "wf_2022": ROOT / "runtime/execution/card/risk_budget_residual_drawdown_guard_walkforward_v1/walkforward/risk_budget_residual_drawdown_guard_walkforward_v1_wf_2022_residual_guard_soxl_daily.csv",
    "wf_2023_2024": ROOT / "runtime/execution/card/risk_budget_residual_drawdown_guard_walkforward_v1/walkforward/risk_budget_residual_drawdown_guard_walkforward_v1_wf_2023_2024_residual_guard_soxl_daily.csv",
}

UPSIDE_FAMILIES = {
    "recovery_reentry_conditional_vol_target_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/recovery_reentry_conditional_vol_target_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/walkforward/wf_2020/recovery_reentry_conditional_vol_target_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/walkforward/wf_2022/recovery_reentry_conditional_vol_target_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/walkforward/wf_2023_2024/recovery_reentry_conditional_vol_target_v1_wf_2023_2024_soxl_daily.csv",
    },
    "recovery_reentry_anchor_confirm_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_anchor_confirm_v1/recovery_reentry_anchor_confirm_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_anchor_confirm_v1/walkforward/wf_2020/recovery_reentry_anchor_confirm_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_anchor_confirm_v1/walkforward/wf_2022/recovery_reentry_anchor_confirm_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_anchor_confirm_v1/walkforward/wf_2023_2024/recovery_reentry_anchor_confirm_v1_wf_2023_2024_soxl_daily.csv",
    },
    "recovery_reentry_regime_drawdown_guard_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/recovery_reentry_regime_drawdown_guard_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/walkforward/wf_2020/recovery_reentry_regime_drawdown_guard_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/walkforward/wf_2022/recovery_reentry_regime_drawdown_guard_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/walkforward/wf_2023_2024/recovery_reentry_regime_drawdown_guard_v1_wf_2023_2024_soxl_daily.csv",
    },
    "recovery_reentry_cppi_floor_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_cppi_floor_v1/recovery_reentry_cppi_floor_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_cppi_floor_v1/walkforward/wf_2020/recovery_reentry_cppi_floor_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_cppi_floor_v1/walkforward/wf_2022/recovery_reentry_cppi_floor_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_cppi_floor_v1/walkforward/wf_2023_2024/recovery_reentry_cppi_floor_v1_wf_2023_2024_soxl_daily.csv",
    },
    "recovery_reentry_2021_trendbreak_cap_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/recovery_reentry_2021_trendbreak_cap_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/walkforward/wf_2020/recovery_reentry_2021_trendbreak_cap_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/walkforward/wf_2022/recovery_reentry_2021_trendbreak_cap_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/walkforward/wf_2023_2024/recovery_reentry_2021_trendbreak_cap_v1_wf_2023_2024_soxl_daily.csv",
    },
}

missing = [str(p.relative_to(ROOT)) for p in BASE_PATHS.values() if not p.exists()]
for family, paths in UPSIDE_FAMILIES.items():
    missing.extend(str(p.relative_to(ROOT)) for p in paths.values() if not p.exists())
if missing:
    raise FileNotFoundError("Missing required daily paths:" + chr(10) + chr(10).join(missing))

print(f"card={CARD.relative_to(ROOT)}")
print(f"upside_families={len(UPSIDE_FAMILIES)}")


card=runtime/reward/card/state_conditioned_rebound_risk_router_v1
upside_families=5


In [2]:
def _coalesce_feature(df: pd.DataFrame, names: list[str], default: float = 0.0) -> pd.Series:
    for name in names:
        if name in df.columns:
            return pd.to_numeric(df[name], errors="coerce")
    return pd.Series(default, index=df.index, dtype=float)


def _load_daily(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.sort_values("datetime").reset_index(drop=True)
    for col in ["price", "position", "strategy_nav", "buyhold_nav"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def _recompute_nav(price: np.ndarray, position: np.ndarray) -> np.ndarray:
    returns = np.zeros_like(price, dtype=float)
    returns[1:] = price[1:] / price[:-1] - 1.0
    applied_position = np.r_[position[0], position[:-1]]
    return STARTING_NAV * np.cumprod(1.0 + applied_position * returns)


def _metrics_from_position(price: np.ndarray, position: np.ndarray) -> dict[str, float]:
    nav = _recompute_nav(price, position)
    buyhold = STARTING_NAV * price / price[0]
    strategy_return = (nav[-1] / nav[0] - 1.0) * 100.0
    buyhold_return = (buyhold[-1] / buyhold[0] - 1.0) * 100.0
    max_dd = (1.0 - nav / np.maximum.accumulate(nav)).max() * 100.0
    turnover = np.abs(np.diff(position, prepend=position[0])).sum()
    return {
        "return": float(strategy_return),
        "buy_and_hold_return": float(buyhold_return),
        "relative_return": float(strategy_return - buyhold_return),
        "max_dd": float(max_dd),
        "avg_pos": float(np.nanmean(position)),
        "turnover": float(turnover),
    }


def _safe_quantile(values: np.ndarray, q: float) -> float:
    clean = values[np.isfinite(values)]
    if len(clean) == 0:
        return float("nan")
    return float(np.quantile(clean, q))


def _build_window(base_path: Path, upside_path: Path, label: str) -> dict:
    base = _load_daily(base_path)
    upside = _load_daily(upside_path)
    keep = ["datetime", "position"]
    merged = base.merge(
        upside[keep].rename(columns={"position": "upside_position"}),
        on="datetime",
        how="inner",
        validate="one_to_one",
    )
    merged = merged.rename(columns={"position": "base_position"})
    # Some base paths already carry diagnostic columns named base_position.
    # Keep the first one, which is the actual executed position renamed above.
    merged = merged.loc[:, ~merged.columns.duplicated()]
    if merged.empty:
        raise ValueError(f"no overlapping dates for {label}: base={len(base)} upside={len(upside)}")
    alignment = {
        "base_rows": int(len(base)),
        "upside_rows": int(len(upside)),
        "merged_rows": int(len(merged)),
        "start": merged["datetime"].iloc[0].strftime("%Y-%m-%d"),
        "end": merged["datetime"].iloc[-1].strftime("%Y-%m-%d"),
        "coverage_truncated": bool(len(merged) != len(base) or len(merged) != len(upside)),
    }

    price_s = pd.to_numeric(merged["price"], errors="coerce")
    price = price_s.to_numpy(dtype=float)
    base_position = pd.to_numeric(merged["base_position"], errors="coerce").to_numpy(dtype=float)
    upside_position = pd.to_numeric(merged["upside_position"], errors="coerce").to_numpy(dtype=float)

    returns_s = price_s.pct_change()
    computed_mom5 = price_s / price_s.shift(5) - 1.0
    computed_mom21 = price_s / price_s.shift(21) - 1.0
    computed_dd63 = price_s / price_s.rolling(63, min_periods=2).max() - 1.0
    computed_rv21 = returns_s.rolling(21, min_periods=5).std() * np.sqrt(252.0)
    computed_vov21 = returns_s.rolling(21, min_periods=5).std()

    mom5 = _coalesce_feature(merged, ["mom_5", "price_mom_5", "lag_mom_5"], default=np.nan).fillna(computed_mom5).fillna(0.0).to_numpy(dtype=float)
    mom21 = _coalesce_feature(merged, ["mom_21", "price_mom_21", "lag_mom_21"], default=np.nan).fillna(computed_mom21).fillna(0.0).to_numpy(dtype=float)
    dd63 = _coalesce_feature(merged, ["drawdown_63", "price_drawdown_63", "lag_drawdown_63"], default=np.nan).fillna(computed_dd63).fillna(0.0).to_numpy(dtype=float)
    rv21 = _coalesce_feature(merged, ["rv_21", "realized_vol_21", "lag_rv_21"], default=np.nan).fillna(computed_rv21).fillna(0.0).to_numpy(dtype=float)
    vol_of_vol_21 = _coalesce_feature(merged, ["vol_of_vol_21", "lag_vol_of_vol_21"], default=np.nan).fillna(computed_vov21).fillna(0.0).to_numpy(dtype=float)
    ratio_widen_21 = _coalesce_feature(merged, ["soxl_soxx_ratio_widen_21", "lag_soxl_soxx_ratio_widen_21"], default=np.nan).fillna(0.0).to_numpy(dtype=float)
    ratio_z_252 = _coalesce_feature(merged, ["soxl_soxx_ratio_z_252", "lag_soxl_soxx_ratio_z_252"], default=np.nan).fillna(0.0).to_numpy(dtype=float)
    soxx_mom21 = _coalesce_feature(merged, ["soxx_mom_21", "lag_soxx_mom_21"], default=np.nan).fillna(0.0).to_numpy(dtype=float)

    base_metrics = _metrics_from_position(price, base_position)
    upside_metrics = _metrics_from_position(price, upside_position)
    base_metrics.update(alignment)
    upside_metrics.update(alignment)
    return {
        "label": label,
        "alignment": alignment,
        "frame": merged,
        "price": price,
        "base_position": base_position,
        "upside_position": upside_position,
        "mom5": mom5,
        "mom21": mom21,
        "dd63": dd63,
        "rv21": rv21,
        "vol_of_vol_21": vol_of_vol_21,
        "ratio_widen_21": ratio_widen_21,
        "ratio_z_252": ratio_z_252,
        "soxx_mom21": soxx_mom21,
        "base_metrics": base_metrics,
        "upside_metrics": upside_metrics,
    }


def _router_position(window: dict, cfg: dict) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    base = window["base_position"]
    upside = window["upside_position"]
    rv21 = window["rv21"]
    vol_of_vol_21 = window["vol_of_vol_21"]
    volatility_signal = np.where(np.isfinite(rv21) & (rv21 > 0), rv21, vol_of_vol_21)
    vol_threshold = _safe_quantile(volatility_signal, cfg["rebound_volq"])

    semis_widen_ok = np.ones_like(base, dtype=bool)
    if cfg["require_semis_widen"]:
        semis_widen_ok = (window["ratio_widen_21"] >= 0.0) | (window["ratio_z_252"] >= -0.25)

    rebound = (
        (window["mom21"] >= cfg["rebound_mom21"])
        & (window["mom5"] >= cfg["rebound_mom5"])
        & (window["dd63"] >= cfg["rebound_min_dd63"])
        & (volatility_signal >= vol_threshold)
        & semis_widen_ok
    )

    broken_anchor_risk = (window["dd63"] <= cfg["risk_dd63"]) & (window["mom21"] <= cfg["risk_mom21"])
    semis_stress = (
        ((window["ratio_z_252"] <= cfg["semis_ratio_z"]) | (window["ratio_widen_21"] <= cfg["semis_ratio_widen"]))
        & (window["soxx_mom21"] <= cfg["semis_soxx_mom21"])
    )
    risk = broken_anchor_risk | (semis_stress if cfg["use_semis_stress"] else False)

    position = base.copy()
    position[risk] = np.minimum(position[risk], cfg["risk_cap"])
    lift = np.maximum(position, np.minimum(upside, cfg["upside_cap"]))
    position[rebound & ~risk] = lift[rebound & ~risk]
    return position, rebound, risk, vol_threshold


def _score_row(metrics_by_window: dict[str, dict]) -> tuple[float, bool, bool]:
    strict = metrics_by_window["strict_oos"]
    strict_pass = strict["relative_return"] >= 30.0 and strict["max_dd"] <= 15.0
    wf_guard = True
    wf_loss = 0.0
    for wf, floor in FULL_PASS_REL_FLOORS.items():
        m = metrics_by_window[wf]
        wf_guard = wf_guard and (m["max_dd"] <= 15.0) and (m["relative_return"] >= floor)
        wf_loss += 2.0 * max(0.0, m["max_dd"] - 15.0) ** 2
        wf_loss += 0.25 * max(0.0, floor - m["relative_return"]) ** 2
    strict_loss = max(0.0, 30.0 - strict["relative_return"]) ** 2 + 4.0 * max(0.0, strict["max_dd"] - 15.0) ** 2
    turnover_loss = 0.02 * strict["turnover"]
    return float(strict_loss + wf_loss + turnover_loss), bool(strict_pass), bool(strict_pass and wf_guard)


In [3]:
windows_by_family: dict[str, dict[str, dict]] = {}
base_metrics_rows = []
upside_metrics_rows = []
for family, paths in UPSIDE_FAMILIES.items():
    windows_by_family[family] = {}
    for label in WINDOWS:
        window = _build_window(BASE_PATHS[label], paths[label], label)
        windows_by_family[family][label] = window
        if family == next(iter(UPSIDE_FAMILIES)):
            base_metrics_rows.append({"family": "base_residual_guard", "window": label, **window["base_metrics"]})
        upside_metrics_rows.append({"family": family, "window": label, **window["upside_metrics"]})

base_metrics_df = pd.DataFrame(base_metrics_rows)
upside_metrics_df = pd.DataFrame(upside_metrics_rows)
base_metrics_df.to_csv(CARD / "base_metrics.csv", index=False)
upside_metrics_df.to_csv(CARD / "upside_metrics.csv", index=False)

display(Markdown("## Base residual guard metrics"))
display(base_metrics_df)
display(Markdown("## Upside candidate metrics"))
display(upside_metrics_df)


## Base residual guard metrics

,family,window,return,buy_and_hold_return,relative_return,max_dd,avg_pos,turnover,base_rows,upside_rows,merged_rows,start,end,coverage_truncated
0,base_residual_guard,strict_oos,176.604683,146.146723,30.457960,14.642461,0.160945,35.149999,1569,1605,1569,2020-01-02,2026-03-31,True
1,base_residual_guard,wf_2020,5.580720,58.093571,-52.512851,18.102212,0.172884,7.175332,253,252,252,2020-01-02,2020-12-30,True
2,base_residual_guard,wf_2022,-9.930770,-86.588072,76.657302,10.970665,0.046000,0.533198,251,250,250,2022-01-03,2022-12-29,True
3,base_residual_guard,wf_2023_2024,35.707210,-0.356635,36.063844,17.229977,0.160577,7.643040,252,251,251,2024-01-02,2024-12-30,True


## Upside candidate metrics

,family,window,return,buy_and_hold_return,relative_return,max_dd,avg_pos,turnover,base_rows,upside_rows,merged_rows,start,end,coverage_truncated
0,recovery_reentry_conditional_vol_target_v1,strict_oos,2443.675249,146.146723,2297.528526,30.923275,0.435293,36.933036,1569,1605,1569,2020-01-02,2026-03-31,True
1,recovery_reentry_conditional_vol_target_v1,wf_2020,113.889962,58.093571,55.796392,26.139320,0.423880,5.614635,253,252,252,2020-01-02,2020-12-30,True
2,recovery_reentry_conditional_vol_target_v1,wf_2022,-21.610018,-86.588072,64.978054,23.626625,0.165095,2.380487,251,250,250,2022-01-03,2022-12-29,True
3,recovery_reentry_conditional_vol_target_v1,wf_2023_2024,68.533647,-0.356635,68.890281,16.826953,0.259105,5.833511,252,251,251,2024-01-02,2024-12-30,True
4,recovery_reentry_anchor_confirm_v1,strict_oos,1542.894521,146.146723,1396.747798,41.587207,0.472879,35.523357,1569,1605,1569,2020-01-02,2026-03-31,True
5,recovery_reentry_anchor_confirm_v1,wf_2020,113.889962,58.093571,55.796392,26.139320,0.423880,5.614635,253,252,252,2020-01-02,2020-12-30,True
6,recovery_reentry_anchor_confirm_v1,wf_2022,-20.683262,-86.588072,65.904809,26.651030,0.183541,2.819416,251,250,250,2022-01-03,2022-12-29,True
7,recovery_reentry_anchor_confirm_v1,wf_2023_2024,58.194746,-0.356635,58.551381,30.183184,0.336830,4.936599,252,251,251,2024-01-02,2024-12-30,True
8,recovery_reentry_regime_drawdown_guard_v1,strict_oos,1854.439444,146.146723,1708.292722,32.120180,0.465440,35.407912,1569,1605,1569,2020-01-02,2026-03-31,True
9,recovery_reentry_regime_drawdown_guard_v1,wf_2020,113.889962,58.093571,55.796392,26.139320,0.423880,5.614635,253,252,252,2020-01-02,2020-12-30,True


In [4]:
GRID = {
    "upside_cap": [0.20, 0.25, 0.35, 0.45],
    "rebound_mom21": [0.30, 0.40, 0.50, 0.70],
    "rebound_volq": [0.40, 0.60, 0.75],
    "risk_cap": [0.00, 0.05, 0.10, 0.15],
    "risk_dd63": [-0.08, -0.10, -0.15, -0.20],
    "risk_mom21": [-0.05, 0.00],
    "use_semis_stress": [False, True],
    "require_semis_widen": [False, True],
}
FIXED = {
    "rebound_mom5": 0.02,
    "rebound_min_dd63": -0.35,
    "semis_ratio_z": -0.75,
    "semis_ratio_widen": -0.05,
    "semis_soxx_mom21": 0.0,
}

def iter_grid():
    keys = list(GRID)
    for values in product(*(GRID[k] for k in keys)):
        cfg = dict(zip(keys, values))
        cfg.update(FIXED)
        yield cfg

rows = []
selected_daily_cache: dict[tuple[str, int, str], pd.DataFrame] = {}
candidate_id = 0
for family, family_windows in windows_by_family.items():
    for cfg in iter_grid():
        metrics_by_window = {}
        state_counts = {}
        vol_thresholds = {}
        positions = {}
        for label, window in family_windows.items():
            position, rebound, risk, vol_threshold = _router_position(window, cfg)
            metrics = _metrics_from_position(window["price"], position)
            metrics_by_window[label] = metrics
            state_counts[label] = {
                "rebound_days": int(rebound.sum()),
                "risk_days": int(risk.sum()),
                "changed_days": int((np.abs(position - window["base_position"]) > 1e-9).sum()),
            }
            vol_thresholds[label] = vol_threshold
            positions[label] = position

        robust_score, strict_pass, full_pass = _score_row(metrics_by_window)
        row = {
            "candidate_id": candidate_id,
            "family": family,
            **{k: cfg[k] for k in GRID.keys()},
            **FIXED,
            "robust_score": robust_score,
            "strict_pass": strict_pass,
            "full_pass": full_pass,
        }
        for label in WINDOWS:
            metrics = metrics_by_window[label]
            row.update({f"{label}_{k}": v for k, v in metrics.items()})
            row.update({f"{label}_{k}": v for k, v in state_counts[label].items()})
            row[f"{label}_vol_threshold"] = vol_thresholds[label]
        rows.append(row)
        candidate_id += 1

screen_df = pd.DataFrame(rows).sort_values(["full_pass", "robust_score"], ascending=[False, True]).reset_index(drop=True)
screen_df.to_csv(CARD / "router_screen.csv", index=False)
top_df = screen_df.head(100).copy()
top_df.to_csv(CARD / "router_top_configs.csv", index=False)

summary_df = (
    screen_df.groupby("family")
    .agg(
        candidates=("candidate_id", "count"),
        strict_pass_count=("strict_pass", "sum"),
        full_pass_count=("full_pass", "sum"),
        best_robust_score=("robust_score", "min"),
        best_strict_relative=("strict_oos_relative_return", "max"),
        best_strict_max_dd=("strict_oos_max_dd", "min"),
    )
    .reset_index()
    .sort_values(["full_pass_count", "best_robust_score"], ascending=[False, True])
)
summary_df.to_csv(CARD / "router_family_summary.csv", index=False)

print(f"screened_candidates={len(screen_df)}")
print(f"strict_pass_count={int(screen_df['strict_pass'].sum())}")
print(f"full_pass_count={int(screen_df['full_pass'].sum())}")
display(summary_df)
display(top_df.head(12))


screened_candidates=30720
strict_pass_count=0
full_pass_count=0


,family,candidates,strict_pass_count,full_pass_count,best_robust_score,best_strict_relative,best_strict_max_dd
2,recovery_reentry_conditional_vol_target_v1,6144,0,0,390.437634,40.486423,14.576033
3,recovery_reentry_cppi_floor_v1,6144,0,0,437.585949,39.132015,14.576033
0,recovery_reentry_2021_trendbreak_cap_v1,6144,0,0,461.373848,39.163098,14.576033
1,recovery_reentry_anchor_confirm_v1,6144,0,0,461.373848,39.163098,14.576033
4,recovery_reentry_regime_drawdown_guard_v1,6144,0,0,461.373848,39.163098,14.576033


,candidate_id,family,upside_cap,rebound_mom21,rebound_volq,risk_cap,risk_dd63,risk_mom21,use_semis_stress,require_semis_widen,...,wf_2023_2024_return,wf_2023_2024_buy_and_hold_return,wf_2023_2024_relative_return,wf_2023_2024_max_dd,wf_2023_2024_avg_pos,wf_2023_2024_turnover,wf_2023_2024_rebound_days,wf_2023_2024_risk_days,wf_2023_2024_changed_days,wf_2023_2024_vol_threshold
0,5118,recovery_reentry_conditional_vol_target_v1,0.45,0.4,0.4,0.15,-0.20,0.0,True,False,...,34.544506,-0.356635,34.901141,17.939129,0.159488,7.889404,3,101,5,0.834176
1,5119,recovery_reentry_conditional_vol_target_v1,0.45,0.4,0.4,0.15,-0.20,0.0,True,True,...,34.544506,-0.356635,34.901141,17.939129,0.159488,7.889404,3,101,5,0.834176
2,5092,recovery_reentry_conditional_vol_target_v1,0.45,0.4,0.4,0.15,-0.08,0.0,False,False,...,38.751230,-0.356635,39.107864,15.373380,0.157935,7.497993,3,109,8,0.834176
3,5093,recovery_reentry_conditional_vol_target_v1,0.45,0.4,0.4,0.15,-0.08,0.0,False,True,...,38.751230,-0.356635,39.107864,15.373380,0.157935,7.497993,3,109,8,0.834176
4,5094,recovery_reentry_conditional_vol_target_v1,0.45,0.4,0.4,0.15,-0.08,0.0,True,False,...,38.751230,-0.356635,39.107864,15.373380,0.157935,7.497993,3,109,8,0.834176
5,5095,recovery_reentry_conditional_vol_target_v1,0.45,0.4,0.4,0.15,-0.08,0.0,True,True,...,38.751230,-0.356635,39.107864,15.373380,0.157935,7.497993,3,109,8,0.834176
6,5102,recovery_reentry_conditional_vol_target_v1,0.45,0.4,0.4,0.15,-0.10,0.0,True,False,...,38.751230,-0.356635,39.107864,15.373380,0.157935,7.497993,3,108,8,0.834176
7,5103,recovery_reentry_conditional_vol_target_v1,0.45,0.4,0.4,0.15,-0.10,0.0,True,True,...,38.751230,-0.356635,39.107864,15.373380,0.157935,7.497993,3,108,8,0.834176
8,5100,recovery_reentry_conditional_vol_target_v1,0.45,0.4,0.4,0.15,-0.10,0.0,False,False,...,38.751230,-0.356635,39.107864,15.373380,0.157935,7.497993,3,108,8,0.834176
9,5101,recovery_reentry_conditional_vol_target_v1,0.45,0.4,0.4,0.15,-0.10,0.0,False,True,...,38.751230,-0.356635,39.107864,15.373380,0.157935,7.497993,3,108,8,0.834176


In [5]:
selected = screen_df.iloc[0].to_dict()
selected_family = selected["family"]
selected_candidate_id = int(selected["candidate_id"])
selected_cfg = {k: selected[k] for k in GRID.keys()}
selected_cfg.update(FIXED)

selected_metrics_rows = []
for label, window in windows_by_family[selected_family].items():
    position, rebound, risk, vol_threshold = _router_position(window, selected_cfg)
    metrics = _metrics_from_position(window["price"], position)
    selected_metrics_rows.append({
        "window": label,
        **metrics,
        "rebound_days": int(rebound.sum()),
        "risk_days": int(risk.sum()),
        "changed_days": int((np.abs(position - window["base_position"]) > 1e-9).sum()),
        "vol_threshold": float(vol_threshold),
    })

    nav = _recompute_nav(window["price"], position)
    out_df = pd.DataFrame({
        "datetime": window["frame"]["datetime"].dt.strftime("%Y-%m-%d"),
        "price": window["price"],
        "buyhold_nav": STARTING_NAV * window["price"] / window["price"][0],
        "strategy_nav": nav,
        "base_position": window["base_position"],
        "upside_position": window["upside_position"],
        "router_position": position,
        "rebound_state": rebound,
        "risk_state": risk,
        "mom_5": window["mom5"],
        "mom_21": window["mom21"],
        "drawdown_63": window["dd63"],
        "rv_21": window["rv21"],
        "vol_of_vol_21": window["vol_of_vol_21"],
        "soxl_soxx_ratio_widen_21": window["ratio_widen_21"],
        "soxl_soxx_ratio_z_252": window["ratio_z_252"],
        "soxx_mom_21": window["soxx_mom21"],
    })
    out_name = "selected_strict_oos_daily.csv" if label == "strict_oos" else f"selected_{label}_daily.csv"
    out_df.to_csv(CARD / out_name, index=False)

selected_metrics_df = pd.DataFrame(selected_metrics_rows)
selected_metrics_df.to_csv(CARD / "selected_metrics.csv", index=False)

def _compact_metrics(row: pd.Series | dict, label: str) -> dict:
    return {
        "return": float(row[f"{label}_return"]),
        "buy_and_hold_return": float(row[f"{label}_buy_and_hold_return"]),
        "relative_return": float(row[f"{label}_relative_return"]),
        "max_dd": float(row[f"{label}_max_dd"]),
        "avg_pos": float(row[f"{label}_avg_pos"]),
        "turnover": float(row[f"{label}_turnover"]),
        "rebound_days": int(row[f"{label}_rebound_days"]),
        "risk_days": int(row[f"{label}_risk_days"]),
        "changed_days": int(row[f"{label}_changed_days"]),
    }

full_pass_count = int(screen_df["full_pass"].sum())
strict_pass_count = int(screen_df["strict_pass"].sum())
coverage_warnings = base_metrics_df.loc[base_metrics_df["coverage_truncated"], ["window", "base_rows", "upside_rows", "merged_rows", "start", "end"]].to_dict("records")
coverage_note = None
if coverage_warnings:
    coverage_note = "Coverage note: the low-drawdown residual base has shorter saved daily coverage than some newer upside paths, so this notebook evaluates common overlapping dates per window. Refreshing the low-drawdown base is a prerequisite before treating this as current 2026-05 production evidence."
if full_pass_count > 0:
    decision = "diagnostic_candidate_for_training_followup"
    conclusion = [
        "At least one fixed-path router clears the strict and walk-forward screen.",
        "Do not promote directly from this notebook; the next step is to implement the state-conditioned objective in training and revalidate out of sample.",
    ]
else:
    decision = "diagnostic_no_promotion"
    blockers = []
    for label in WINDOWS:
        m = selected_metrics_df.loc[selected_metrics_df["window"] == label].iloc[0]
        if m["max_dd"] > 15.0:
            blockers.append(f"{label} MaxDD {m['max_dd']:.2f}% > 15%")
        floor = 30.0 if label == "strict_oos" else FULL_PASS_REL_FLOORS[label]
        if m["relative_return"] < floor:
            blockers.append(f"{label} relative {m['relative_return']:.2f}pp < {floor:.2f}pp floor")
    conclusion = [
        "No screened state-conditioned fixed-path router clears the full strict plus walk-forward gate.",
        "The selected lowest-loss router remains useful as a mechanism diagnostic, not a production candidate.",
        "Primary blockers: " + "; ".join(blockers[:6]),
        "Next experiment should avoid another scalar/global reward sweep and instead train or select with explicit state-conditioned terms plus walk-forward checkpoint weighting.",
    ]
if coverage_note:
    conclusion.insert(2, coverage_note)

selection = {
    "experiment": "state_conditioned_rebound_risk_router_v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "changed_layer": ["reward", "selection"],
    "candidate_count": int(len(screen_df)),
    "strict_pass_count": strict_pass_count,
    "full_pass_count": full_pass_count,
    "selected_candidate_id": selected_candidate_id,
    "selected_family": selected_family,
    "selected_config": selected_cfg,
    "coverage_warnings": coverage_warnings,
    "selected_metrics": {row["window"]: {k: row[k] for k in row.index if k != "window"} for _, row in selected_metrics_df.iterrows()},
    "selected_screen_row": {k: (v.item() if hasattr(v, "item") else v) for k, v in selected.items()},
    "external_sources": [
        "https://github.com/microsoft/qlib",
        "https://ojs.aaai.org/index.php/AAAI/article/view/16144",
        "https://www.sciencedirect.com/science/article/pii/S0957417417304402",
        "https://github.com/AI4Finance-Foundation/FinRL",
    ],
    "conclusion": conclusion,
    "artifacts": {
        "notebook": "runtime/reward/card/state_conditioned_rebound_risk_router_v1/state_conditioned_rebound_risk_router_v1.ipynb",
        "base_metrics": "runtime/reward/card/state_conditioned_rebound_risk_router_v1/base_metrics.csv",
        "upside_metrics": "runtime/reward/card/state_conditioned_rebound_risk_router_v1/upside_metrics.csv",
        "router_screen": "runtime/reward/card/state_conditioned_rebound_risk_router_v1/router_screen.csv",
        "router_top_configs": "runtime/reward/card/state_conditioned_rebound_risk_router_v1/router_top_configs.csv",
        "router_family_summary": "runtime/reward/card/state_conditioned_rebound_risk_router_v1/router_family_summary.csv",
        "selected_metrics": "runtime/reward/card/state_conditioned_rebound_risk_router_v1/selected_metrics.csv",
        "selected_daily": [
            "runtime/reward/card/state_conditioned_rebound_risk_router_v1/selected_strict_oos_daily.csv",
            "runtime/reward/card/state_conditioned_rebound_risk_router_v1/selected_wf_2020_daily.csv",
            "runtime/reward/card/state_conditioned_rebound_risk_router_v1/selected_wf_2022_daily.csv",
            "runtime/reward/card/state_conditioned_rebound_risk_router_v1/selected_wf_2023_2024_daily.csv",
        ],
        "selection": "runtime/reward/card/state_conditioned_rebound_risk_router_v1/selection.json",
    },
}
(CARD / "selection.json").write_text(json.dumps(selection, indent=2, ensure_ascii=False) + chr(10), encoding="utf-8")

display(Markdown("## Selected Router Metrics"))
display(selected_metrics_df)
display(Markdown("## Conclusion"))
for item in conclusion:
    display(Markdown(f"- {item}"))
print(json.dumps({
    "decision": decision,
    "selected_family": selected_family,
    "selected_candidate_id": selected_candidate_id,
    "strict_pass_count": strict_pass_count,
    "full_pass_count": full_pass_count,
}, indent=2))


## Selected Router Metrics

,window,return,buy_and_hold_return,relative_return,max_dd,avg_pos,turnover,rebound_days,risk_days,changed_days,vol_threshold
0,strict_oos,173.038419,146.146723,26.891697,15.715202,0.161419,41.414824,60,601,154,0.866011
1,wf_2020,10.523972,58.093571,-47.569598,16.969277,0.176973,8.073416,11,56,23,0.849744
2,wf_2022,-9.503019,-86.588072,77.085053,10.547853,0.045775,0.430146,4,169,3,1.225981
3,wf_2023_2024,34.544506,-0.356635,34.901141,17.939129,0.159488,7.889404,3,101,5,0.834176


## Conclusion

- No screened state-conditioned fixed-path router clears the full strict plus walk-forward gate.

- The selected lowest-loss router remains useful as a mechanism diagnostic, not a production candidate.

- Coverage note: the low-drawdown residual base has shorter saved daily coverage than some newer upside paths, so this notebook evaluates common overlapping dates per window. Refreshing the low-drawdown base is a prerequisite before treating this as current 2026-05 production evidence.

- Primary blockers: strict_oos MaxDD 15.72% > 15%; strict_oos relative 26.89pp < 30.00pp floor; wf_2020 MaxDD 16.97% > 15%; wf_2020 relative -47.57pp < -10.00pp floor; wf_2023_2024 MaxDD 17.94% > 15%

- Next experiment should avoid another scalar/global reward sweep and instead train or select with explicit state-conditioned terms plus walk-forward checkpoint weighting.

{
  "decision": "diagnostic_no_promotion",
  "selected_family": "recovery_reentry_conditional_vol_target_v1",
  "selected_candidate_id": 5118,
  "strict_pass_count": 0,
  "full_pass_count": 0
}
